### Base Model Training

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import pandas as pd

In [3]:

# Load the sampled dataset
df = pd.read_csv("/content/drive/MyDrive/Road_Accident_Aalysis_project/Accidents_Cleaned.csv")
df.head()

,Severity,Start_Lat,Start_Lng,Distance(mi),City,County,State,Zipcode,Country,Timezone,...,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Duration_Minutes,Hour,DayOfWeek,Month,IsWeekend,IsDay
0,2,27.843691,-82.811668,0.00,Seminole,Pinellas,FL,33772,US,US/Eastern,...,0,0,1,0,29.816667,7,4,11,0,1
1,2,38.420071,-77.423340,0.00,Stafford,Stafford,VA,22554,US,US/Eastern,...,0,0,0,0,74.716667,7,2,12,0,1
2,2,42.319832,-71.226318,0.01,Waban,Middlesex,MA,02468-2321,US,US/Eastern,...,0,0,0,0,29.700000,7,0,12,0,1
3,2,39.034676,-84.599693,0.00,Erlanger,Kenton,KY,41018,US,US/Eastern,...,0,0,0,0,61.416667,6,0,2,0,0
4,2,43.010193,-83.689369,0.00,Flint,Genesee,MI,48502-1051,US,US/Eastern,...,0,0,1,0,81.016667,11,3,8,0,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 977587 entries, 0 to 977586
Data columns (total 40 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Severity           977587 non-null  int64  
 1   Start_Lat          977587 non-null  float64
 2   Start_Lng          977587 non-null  float64
 3   Distance(mi)       977587 non-null  float64
 4   City               977587 non-null  object 
 5   County             977587 non-null  object 
 6   State              977587 non-null  object 
 7   Zipcode            977587 non-null  object 
 8   Country            977587 non-null  object 
 9   Timezone           977587 non-null  object 
 10  Airport_Code       977587 non-null  object 
 11  Weather_Timestamp  977587 non-null  object 
 12  Temperature(F)     977587 non-null  float64
 13  Wind_Chill(F)      977587 non-null  float64
 14  Humidity(%)        977587 non-null  float64
 15  Pressure(in)       977587 non-null  float64
 16  Vi

In [5]:
df

,Severity,Start_Lat,Start_Lng,Distance(mi),City,County,State,Zipcode,Country,Timezone,...,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Duration_Minutes,Hour,DayOfWeek,Month,IsWeekend,IsDay
0,2,27.843691,-82.811668,0.00,Seminole,Pinellas,FL,33772,US,US/Eastern,...,0,0,1,0,29.816667,7,4,11,0,1
1,2,38.420071,-77.423340,0.00,Stafford,Stafford,VA,22554,US,US/Eastern,...,0,0,0,0,74.716667,7,2,12,0,1
2,2,42.319832,-71.226318,0.01,Waban,Middlesex,MA,02468-2321,US,US/Eastern,...,0,0,0,0,29.700000,7,0,12,0,1
3,2,39.034676,-84.599693,0.00,Erlanger,Kenton,KY,41018,US,US/Eastern,...,0,0,0,0,61.416667,6,0,2,0,0
4,2,43.010193,-83.689369,0.00,Flint,Genesee,MI,48502-1051,US,US/Eastern,...,0,0,1,0,81.016667,11,3,8,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
977582,2,35.223316,-80.629066,0.00,Mint Hill,Mecklenburg,NC,28227-6871,US,US/Eastern,...,0,0,1,0,96.383333,9,3,11,0,1
977583,3,40.727837,-73.928467,0.00,Maspeth,Queens,NY,11378,US,US/Eastern,...,0,0,0,0,44.550000,15,1,11,0,1
977584,2,35.202499,-80.725975,0.00,Charlotte,Mecklenburg,NC,28227,US,US/Eastern,...,0,0,1,0,61.800000,11,0,6,0,1
977585,2,33.846607,-118.205360,0.00,Long Beach,Los Angeles,CA,90805,US,US/Pacific,...,0,0,0,0,78.750000,1,0,10,0,0


In [6]:
df.isnull().sum()

,0
Severity,0
Start_Lat,0
Start_Lng,0
Distance(mi),0
City,0
County,0
State,0
Zipcode,0
Country,0
Timezone,0


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.impute import SimpleImputer

In [8]:
# Define target and features
target = 'Severity'
X = df.drop(columns=[target])
y = df[target]

In [9]:
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64', 'bool']).columns.tolist()

In [10]:
# Preprocessing pipelines for numeric and categorical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # fill missing numerics if any
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # fill missing categoricals if any
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [11]:
# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [12]:
# Create pipeline with preprocessing and logistic regression
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='auto', n_jobs=-1))
])

In [13]:
# Split dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_train.shape

(782069, 39)

In [16]:
X_train_small = X_train.sample(n=100000, random_state=42)
y_train_small = y_train.loc[X_train_small.index]

clf.fit(X_train_small, y_train_small)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Start_Lat', 'Start_Lng',
                                                   'Distance(mi)',
                                                   'Temperature(F)',
                                                   'Wind_Chill(F)',
                                                   'Humidity(%)',
                                                   'Pressure(in)',
                                                   'Visibility(mi)',
                                                   'Wind_Speed(mph)',
                                                   'Precipitation(in)',
                                                   'Amenity', 'Bump',
                                                   'Crossing', 'Gi...
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['City', 'County', 'State',
                                                   'Zipcode', 'Country',
                                                   'Timezone', 'Airport_Code',
                                                   'Weather_Timestamp',
                                                   'Wind_Direction',
                                                   'Weather_Condition'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, multi_class='auto',
                                    n_jobs=-1))])

In [17]:
# Predict on test
y_pred = clf.predict(X_test)

In [18]:
# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8151679129287329
Classification Report:
               precision    recall  f1-score   support

           1       0.54      0.05      0.10      3817
           2       0.85      0.88      0.87    126980
           3       0.75      0.74      0.74     63548
           4       0.22      0.03      0.06      1173

    accuracy                           0.82    195518
   macro avg       0.59      0.43      0.44    195518
weighted avg       0.81      0.82      0.81    195518



### Task:
- Train `DecisionTree` Classification model using this cleaned data and pipeline, and report the metric evaluation.

In [20]:
from sklearn.tree import DecisionTreeClassifier

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=10, min_samples_split=20))
])


In [21]:
# Fit the model
X_train_small = X_train.sample(n=100000, random_state=42)
y_train_small = y_train.loc[X_train_small.index]

clf.fit(X_train_small, y_train_small)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Start_Lat', 'Start_Lng',
                                                   'Distance(mi)',
                                                   'Temperature(F)',
                                                   'Wind_Chill(F)',
                                                   'Humidity(%)',
                                                   'Pressure(in)',
                                                   'Visibility(mi)',
                                                   'Wind_Speed(mph)',
                                                   'Precipitation(in)',
                                                   'Amenity', 'Bump',
                                                   'Crossing', 'Gi...
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['City', 'County', 'State',
                                                   'Zipcode', 'Country',
                                                   'Timezone', 'Airport_Code',
                                                   'Weather_Timestamp',
                                                   'Wind_Direction',
                                                   'Weather_Condition'])])),
                ('classifier',
                 DecisionTreeClassifier(max_depth=10, min_samples_split=20))])

In [22]:
# Predict on test
y_pred = clf.predict(X_test)

In [23]:
# Evaluate (Decision Tree)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7130187501917982
Classification Report:
               precision    recall  f1-score   support

           1       0.27      0.01      0.01      3817
           2       0.78      0.78      0.78    126980
           3       0.58      0.63      0.60     63548
           4       0.45      0.07      0.11      1173

    accuracy                           0.71    195518
   macro avg       0.52      0.37      0.38    195518
weighted avg       0.71      0.71      0.71    195518

